# Credit Card Fraud Detection — Notebook Guide

## What this project does
A credit card transaction is labelled **fraudulent** (`is_fraud = 1`) or **legitimate** (`is_fraud = 0`). This notebook builds a machine learning pipeline that *learns from historical transactions* to automatically predict fraud on future ones — the goal is to identify suspicious activity **before** it's confirmed, not just label it afterwards.

## The dataset
- **Source**: Kaggle "kartik2112/fraud-detection" — a realistic synthetic simulation of real spending behavior
- **Train**: ~1.3 M transactions (Jan 2019 – Jun 2020)
- **Test**: ~556 K transactions (Jun 2020 – Dec 2020)
- **Target column**: `is_fraud` — only ~0.58% of transactions are fraudulent (highly imbalanced)
- **23 columns**: transaction amount, merchant category, location coordinates, customer demographics, timestamps

## How to use this notebook
> Run cells **top to bottom** in order — each step builds on variables defined in the one before it.
> If a cell fails with a `NameError`, it usually means an earlier cell was not run yet.

## Pipeline map — 8 steps at a glance

| Step | Cell(s) | What it does |
|------|---------|--------------|
| 1 | Data Profiling | Load data, validate schema, check nulls and temporal split |
| 2 | Imbalance Analysis | Measure how rare fraud is; choose appropriate evaluation metrics |
| 3 | Drift Analysis | Check whether transaction patterns changed between train and test periods |
| 4 | Fraud Pattern Mining | Find which categories, hours, and amounts carry the most fraud risk |
| 5 | Leakage Analysis | Identify raw columns that would unfairly "give away" the answer |
| 6 | Feature Strategy | Decide exactly which columns to keep, drop, or engineer |
| 7 | Feature Engineering + Preprocessing | Build new features, encode categories, scale numerics |
| 8 | Baseline Modeling | Train first models and compare them with fraud-appropriate metrics |

> **Key idea**: accuracy is misleading here. A model that always predicts "not fraud" would be 99.4% accurate yet catch **zero** fraud cases. We use **PR-AUC** (precision-recall area under curve) as our primary metric instead.


In [ ]:
# =============================================================================
# COLAB SETUP - Run this cell first!
# =============================================================================
# Install missing dependencies (Colab has most, but not all)
!pip install -q imbalanced-learn plotly

# Download dataset from Kaggle
# You need a Kaggle API key: go to kaggle.com -> Account -> Create New Token
# Then upload your kaggle.json when prompted below.
import os
if not os.path.exists('Data/fraudTrain.csv'):
    from google.colab import files
    print('Upload your kaggle.json file:')
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip install -q kaggle
    !kaggle datasets download -d kartik2112/fraud-detection --force
    !mkdir -p Data && unzip -o fraud-detection.zip -d Data/
    print('Dataset downloaded and extracted to Data/')
else:
    print('Dataset already present.')

In [1]:
# =============================================================================
# DATA HANDLING
# pandas: the primary tool for loading, filtering, and transforming tabular data.
# numpy: fast numerical arrays and math functions (used under the hood by pandas/sklearn).
# =============================================================================
import pandas as pd
import numpy as np

# =============================================================================
# VISUALIZATION
# matplotlib / seaborn: static charts (distributions, confusion matrices, heatmaps).
# plotly: interactive charts where you can hover to inspect values — useful for EDA.
# =============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# =============================================================================
# PREPROCESSING UTILITIES
# StandardScaler: rescales numeric features to mean=0, std=1 so no single
#   feature dominates the model due to its units (e.g. city_pop in thousands
#   vs. amt in dollars would otherwise make amt look unimportant).
# LabelEncoder: converts text categories ("shopping_net", "grocery_pos") to
#   integers (0, 1, 2, ...) so models can process them numerically.
# train_test_split: splits a dataset into training and validation subsets.
#   Not used here (we have a pre-split temporal dataset) but available for experiments.
# =============================================================================
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# =============================================================================
# IMBALANCED DATA HANDLING
# SMOTE (Synthetic Minority Oversampling Technique): generates synthetic fraud
#   examples to balance the dataset. Not used in the current pipeline (we use
#   class_weight instead) but imported for potential experimentation.
# =============================================================================
from imblearn.over_sampling import SMOTE

# =============================================================================
# CLASSICAL ML MODELS
# All follow the same scikit-learn API: .fit() to train, .predict() to classify.
# LogisticRegression: linear baseline — fast, interpretable, good sanity check.
# RandomForestClassifier: ensemble of decision trees — handles non-linearity well.
# GradientBoostingClassifier: sequential tree boosting — powerful but slower.
# DecisionTreeClassifier: single tree — useful for visualizing decision logic.
# =============================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

# =============================================================================
# EVALUATION METRICS
# classification_report: prints precision, recall, and F1 for each class.
# confusion_matrix / ConfusionMatrixDisplay: shows TP, FP, FN, TN counts visually.
# roc_auc_score / roc_curve: measures overall ranking quality (AUC = 1.0 is perfect).
# =============================================================================
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay
)

# =============================================================================
# DEEP LEARNING STACK
# TensorFlow / Keras: builds neural networks layer by layer.
# Included as an optional alternative baseline — not used in the main pipeline yet.
# =============================================================================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# =============================================================================
# MODEL PERSISTENCE
# joblib: saves a trained model object to disk so it can be reloaded later
#   without retraining. Faster than pickle for large NumPy arrays.
# =============================================================================
import joblib

# Suppress low-priority warnings (convergence hints, future deprecations).
# This keeps notebook output clean for teaching and demonstration purposes.
# Tradeoff: you may miss useful diagnostic hints when debugging unusual behavior.
# To investigate unexpected results, comment out the line below.
import warnings
warnings.filterwarnings('ignore')

# Printing library versions creates a "reproducibility contract":
# anyone running this notebook can verify they have the same software environment.
# Version mismatches are a common source of subtle, hard-to-diagnose bugs.
print(f'pandas     {pd.__version__}')
print(f'numpy      {np.__version__}')
print(f'sklearn    {__import__("sklearn").__version__}')
print(f'tensorflow {tf.__version__}')
print(f'seaborn    {sns.__version__}')


pandas     2.2.3
numpy      2.1.3
sklearn    1.6.1
tensorflow 2.20.0
seaborn    0.13.2


## Step 1: Data Profiling

### Goal
Understand the basic structure and quality of the dataset before doing any modeling.

### Key concepts for beginners

**What is a CSV file?**
A CSV (Comma-Separated Values) file stores data as plain text — each row is one record (transaction), and columns are separated by commas. `index_col=0` tells pandas to treat the first column as the row index rather than data. Kaggle datasets automatically export a numbered index column we don't need as a feature.

**What does `.shape` mean?**
`df.shape` returns `(rows, columns)`. For example, `(1296675, 22)` means 1,296,675 transactions, each described by 22 pieces of information. Rows = individual transactions. Columns = features we can use to predict fraud.

**Why a temporal split?**
In real fraud detection, you train on historical transactions and predict on future ones. If test data "leaks" into training (e.g., a June transaction appears in a training set labeled "Jan–May"), the model appears to learn but is actually seeing the future. The check `max(train_date) < min(test_date)` ensures every training transaction happened *before* every test transaction. Think of it as: *you can only predict the future using the past.*

**What is a null value?**
A null (or `NaN` — "Not a Number") means a value is missing from a cell. If a column has many nulls, the model can't use those rows without handling it first — either by filling in an estimate (imputation) or by dropping the affected rows. Checking for nulls early lets us decide on a strategy before modeling.

### What this step checks
- number of rows and columns
- whether train and test have the same schema
- missing values in train and test
- class balance (`is_fraud`)
- time coverage of train/test

### How to read the output
- If schema columns do not match, stop and fix the data pipeline.
- If temporal split check fails, stop (this would cause leakage).
- If null counts are high, decide on imputation or feature removal.

### Why this matters
A fraud model is only reliable if input data is clean, consistent, and leakage-safe.

### What to report
- Train/Test size
- Fraud rate in each split
- Temporal boundary between train and test
- Whether nulls/schema checks passed


In [2]:
# ---------------------------------------------------------------------------
# Step 1: Data loading and profiling checks
# ---------------------------------------------------------------------------
# Goal: make sure train/test files are structurally valid before any mining.

TRAIN_PATH = "Data/fraudTrain.csv"
TEST_PATH  = "Data/fraudTest.csv"

# index_col=0 removes the unnamed CSV index column Kaggle adds automatically.
# Without it, pandas creates a column called "Unnamed: 0" containing row numbers
# (0, 1, 2, ...) — noise that looks like data and would confuse downstream models.
df_train = pd.read_csv(TRAIN_PATH, index_col=0)
df_test  = pd.read_csv(TEST_PATH,  index_col=0)

# Basic health checks.
print(f"Train shape : {df_train.shape}")
print(f"Test  shape : {df_test.shape}")
print(f"\nFraud rate (train): {df_train['is_fraud'].mean():.4%}")
print(f"Fraud rate (test) : {df_test['is_fraud'].mean():.4%}")

# Schema consistency check: a model trained on columns [A, B, C] cannot make
# predictions on data with columns [A, B, D] — the feature space must match exactly.
# `assert` is a debugging safety net: it crashes immediately with a clear message
# if the condition is False, rather than silently producing wrong results later.
train_cols = df_train.columns.tolist()
test_cols = df_test.columns.tolist()
assert train_cols == test_cols, "Schema mismatch between train and test columns"
print(f"\nSchema check: {len(train_cols)} columns match between train and test")
print(f"Columns:\n{train_cols}")

# isnull() returns a True/False mask for every cell (True = value is missing).
# .sum() counts the True values per column (Python treats True as 1, False as 0).
# Filtering to [> 0] keeps the output clean: only problem columns are shown.
train_nulls = df_train.isnull().sum()
test_nulls = df_test.isnull().sum()
print(f"\nNull counts (train):\n{train_nulls[train_nulls > 0]}")
print(f"\nNull counts (test):\n{test_nulls[test_nulls > 0]}")

# Temporal holdout check: all train data must come before all test data.
# This prevents leakage from future behavior into training.
# max(train) < min(test) means: the latest training transaction is still
# earlier than the earliest test transaction — no time-based overlap exists.
train_ts_max = pd.to_datetime(df_train["trans_date_trans_time"]).max()
test_ts_min = pd.to_datetime(df_test["trans_date_trans_time"]).min()
assert train_ts_max < test_ts_min, (
    f"Temporal split violation: max(train)={train_ts_max} is not before min(test)={test_ts_min}"
)
print(f"\nTemporal split check: max(train)={train_ts_max} < min(test)={test_ts_min}")


Train shape : (1296675, 22)
Test  shape : (555719, 22)

Fraud rate (train): 0.5789%
Fraud rate (test) : 0.3860%

Schema check: 22 columns match between train and test
Columns:
['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']

Null counts (train):
Series([], dtype: int64)

Null counts (test):
Series([], dtype: int64)

Temporal split check: max(train)=2020-06-21 12:13:37 < min(test)=2020-06-21 12:14:25


## Step 2: Class Imbalance Analysis

### Goal
Measure how rare fraud cases are, and quantify imbalance severity.

### Key concepts for beginners

**The "lazy model" thought experiment**
Imagine a model that always predicts "not fraud" for every single transaction — it never raises a single alarm. On this dataset, with only 0.58% fraud, that model would be **99.4% accurate**. Yet it catches exactly **zero fraud cases**. This is why accuracy is a misleading metric for imbalanced problems: doing nothing looks impressive by accuracy alone.

**Better metrics: ROC-AUC and PR-AUC**
- **ROC-AUC** (Receiver Operating Characteristic — Area Under Curve): measures how well the model *ranks* fraud cases above non-fraud cases. A score of 1.0 is perfect; 0.5 is random guessing. Good for measuring overall discrimination ability.
- **PR-AUC** (Precision-Recall — Area Under Curve): focuses on the performance in the fraud-relevant score range. Much more sensitive to rare-class detection than ROC-AUC. **This is our primary metric** for comparing models.

**Precision and Recall — plain English**
- **Precision**: "Of all transactions I flagged as fraud, what fraction were *actually* fraud?" High precision = few false alarms.
- **Recall**: "Of all *actual* fraud cases in the data, what fraction did I catch?" High recall = few missed frauds.
- They trade off against each other: being more aggressive catches more fraud (higher recall) but also flags more innocent transactions (lower precision).

**SMOTE — a preview**
SMOTE (Synthetic Minority Oversampling Technique) handles imbalance by generating *synthetic* fraud examples so the model trains on a more balanced mix. This notebook uses `class_weight="balanced"` instead — a simpler approach that re-weights the loss function without creating new data. SMOTE is imported for experimentation.

### What this step checks
- `fraud_count` and `non_fraud_count`
- `fraud_rate` and `non_fraud_rate`
- `imbalance_ratio = non_fraud / fraud`
- fraud-rate shift from train to test

### How to read the output
- Very large imbalance ratio means accuracy is not a useful metric.
- A train/test fraud-rate shift means threshold calibration may need adjustment.

### Why this matters
Fraud detection is a rare-event task. Imbalance drives model choice, metric choice, and threshold policy.

### What to report
- Imbalance ratio in train and test
- Fraud-rate shift (percentage points)
- Modeling implication: use class weighting and precision-recall metrics


In [3]:
# ---------------------------------------------------------------------------
# Step 2: Class imbalance analysis
# ---------------------------------------------------------------------------
# Goal: quantify rare-event imbalance and document modeling implications.

TARGET = "is_fraud"
VALID_TARGET_VALUES = {0, 1}

# Guardrail: target must stay binary in both splits.
train_target_values = set(df_train[TARGET].dropna().unique())
test_target_values = set(df_test[TARGET].dropna().unique())
assert train_target_values.issubset(VALID_TARGET_VALUES), (
    f"Unexpected target values in train: {train_target_values}"
)
assert test_target_values.issubset(VALID_TARGET_VALUES), (
    f"Unexpected target values in test: {test_target_values}"
)


def summarize_imbalance(df, split_name, target_col=TARGET):
    """Return and print core imbalance metrics for one split."""
    # value_counts() counts how many 0s (legitimate) and 1s (fraud) exist.
    # sort_index() ensures class 0 always appears before class 1, regardless
    # of which class is more common (important for consistent downstream logic).
    counts = df[target_col].value_counts().sort_index()

    # Safe .get(key, default) avoids KeyError if one class is entirely absent.
    # For example, if a split somehow had no fraud at all, counts.get(1, 0)
    # returns 0 instead of crashing with a missing-key error.
    non_fraud = int(counts.get(0, 0))
    fraud = int(counts.get(1, 0))
    total = non_fraud + fraud

    fraud_rate = fraud / total if total else 0.0
    non_fraud_rate = non_fraud / total if total else 0.0

    # np.inf (infinity) guards against division by zero when fraud count = 0.
    # It signals "infinitely imbalanced" — conceptually correct and avoids a crash.
    imbalance_ratio = (non_fraud / fraud) if fraud else np.inf

    print(f"\n[{split_name}]")
    print(f"  total            : {total:,}")
    print(f"  non_fraud_count  : {non_fraud:,}")
    print(f"  fraud_count      : {fraud:,}")
    print(f"  non_fraud_rate   : {non_fraud_rate:.4%}")
    print(f"  fraud_rate       : {fraud_rate:.4%}")
    print(f"  imbalance_ratio  : {imbalance_ratio:.2f}:1 (non_fraud:fraud)")

    return {
        "split": split_name,
        "total": total,
        "non_fraud": non_fraud,
        "fraud": fraud,
        "non_fraud_rate": non_fraud_rate,
        "fraud_rate": fraud_rate,
        "imbalance_ratio": imbalance_ratio,
    }


imbalance_train = summarize_imbalance(df_train, "Train")
imbalance_test = summarize_imbalance(df_test, "Test")

# Positive value means fraud is more frequent in test than train.
fraud_rate_shift_pp = (imbalance_test["fraud_rate"] - imbalance_train["fraud_rate"]) * 100
print(f"\nFraud-rate shift (test - train): {fraud_rate_shift_pp:.4f} percentage points")

print("\nModeling implication (precision-first): prefer class-weighted training and threshold tuning over accuracy.")



[Train]
  total            : 1,296,675
  non_fraud_count  : 1,289,169
  fraud_count      : 7,506
  non_fraud_rate   : 99.4211%
  fraud_rate       : 0.5789%
  imbalance_ratio  : 171.75:1 (non_fraud:fraud)

[Test]
  total            : 555,719
  non_fraud_count  : 553,574
  fraud_count      : 2,145
  non_fraud_rate   : 99.6140%
  fraud_rate       : 0.3860%
  imbalance_ratio  : 258.08:1 (non_fraud:fraud)

Fraud-rate shift (test - train): -0.1929 percentage points

Modeling implication (precision-first): prefer class-weighted training and threshold tuning over accuracy.


## Step 3: Drift Analysis (Train vs Test)

### Goal
Check whether transaction behavior changed between training period and test period.

### Key concepts for beginners

**What is data drift?**
Imagine training a fraud model on transactions from January, then deploying it in July. Shopping patterns change: holiday spending ends, summer travel begins, average amounts shift. The model learned to recognize January fraud — but now it's seeing July transactions. *Data drift* is when the statistical distribution of features changes between the time you trained the model and the time you use it. If drift is large enough, model predictions become unreliable.

**PSI (Population Stability Index)**
PSI is a standard industry metric for measuring how much a feature's distribution changed between two time periods. It works by:
1. Dividing the training distribution into 10 equal-frequency buckets (quantiles)
2. Counting what fraction of test observations fall in each bucket
3. Computing a weighted sum of log-ratios between train and test bucket shares

| PSI value | Risk level | Meaning |
|-----------|-----------|---------|
| < 0.10    | Low       | Distribution is stable — safe to use as-is |
| 0.10–0.25 | Medium    | Noticeable shift — monitor model performance carefully |
| ≥ 0.25    | High      | Major shift — consider retraining or feature adjustment |

**Why does `unix_time` show PSI = 11.51 (extremely high)?**
`unix_time` is a raw Unix timestamp — it records *when* each transaction happened (seconds elapsed since January 1, 1970). The test set is 6 months later than the train set, so of course these numbers are completely different. Every single test timestamp is larger than every single train timestamp. This extreme PSI is expected and not alarming. It is precisely why `unix_time` is flagged for "engineer then drop": we extract the useful signals (hour of day, day of week) and discard the raw timestamp that encodes time ordering.

### What this step checks
- monthly fraud-rate trends
- numeric drift using PSI (Population Stability Index)
- categorical share shifts (`abs_delta`) for key features

### How to read the output
- PSI `< 0.10`: low drift
- PSI `0.10 - 0.25`: medium drift
- PSI `>= 0.25`: high drift
- Large categorical share deltas indicate unstable feature behavior.

### Why this matters
If data distribution changes, model quality can drop after deployment.

### What to report
- highest-drift numeric features (with PSI)
- biggest categorical shifts
- whether drift is low/medium/high overall


In [4]:
# ---------------------------------------------------------------------------
# Step 3: Drift analysis (Train vs Test)
# ---------------------------------------------------------------------------
# Goal: detect distribution changes that can reduce model reliability.

TARGET = "is_fraud"


def population_stability_index(expected, actual, bins=10, eps=1e-6):
    """Calculate PSI using quantile bins from the expected (train) distribution."""
    expected = pd.Series(expected).dropna().astype(float)
    actual = pd.Series(actual).dropna().astype(float)

    if expected.empty or actual.empty:
        return np.nan

    # np.linspace(0, 1, bins+1) creates 11 evenly spaced thresholds from 0% to 100%.
    # These become percentile cutpoints: [0%, 10%, 20%, ..., 100%].
    # Using TRAIN quantiles as bin edges is standard PSI practice — it anchors the
    # baseline (train) distribution so deviations in test are easy to detect.
    q = np.linspace(0, 1, bins + 1)
    breaks = expected.quantile(q).to_numpy()
    # np.unique removes duplicate edges that arise when a column has very few
    # distinct values (e.g. a binary flag produces only two unique quantile values).
    breaks = np.unique(breaks)

    # Fallback for low-variance columns: if fewer than 3 unique quantile edges
    # exist, np.histogram would crash or produce nonsensical bins. We fall back
    # to evenly spaced bins across the full observed range instead.
    if breaks.size < 3:
        min_v = min(expected.min(), actual.min())
        max_v = max(expected.max(), actual.max())
        if min_v == max_v:
            return 0.0  # constant column — no drift is possible
        breaks = np.linspace(min_v, max_v, bins + 1)

    exp_counts, _ = np.histogram(expected, bins=breaks)
    act_counts, _ = np.histogram(actual, bins=breaks)

    exp_pct = exp_counts / max(exp_counts.sum(), 1)
    act_pct = act_counts / max(act_counts.sum(), 1)

    # np.clip(arr, eps, None) replaces any 0.0 with eps (a tiny positive number).
    # This is required because np.log(0) = -infinity, which would corrupt the PSI sum.
    # Using eps=1e-6 changes the result by a negligible amount for non-empty bins.
    exp_pct = np.clip(exp_pct, eps, None)
    act_pct = np.clip(act_pct, eps, None)

    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))


def psi_risk_band(psi):
    """Map PSI to common risk buckets for reporting."""
    if pd.isna(psi):
        return "unknown"
    if psi >= 0.25:
        return "high"
    if psi >= 0.10:
        return "medium"
    return "low"


# 1) Temporal drift: compare monthly fraud rates between splits.
# dt.to_period("M") converts a full timestamp like "2020-06-15 14:32:00"
# to a year-month label like "2020-06", grouping all transactions in the same
# calendar month together for trend comparison.
train_month = pd.to_datetime(df_train["trans_date_trans_time"]).dt.to_period("M").astype(str)
test_month = pd.to_datetime(df_test["trans_date_trans_time"]).dt.to_period("M").astype(str)

train_monthly = (
    pd.DataFrame({"month": train_month, TARGET: df_train[TARGET]})
    .groupby("month", as_index=False)[TARGET]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"count": "tx_count", "mean": "fraud_rate"})
)

test_monthly = (
    pd.DataFrame({"month": test_month, TARGET: df_test[TARGET]})
    .groupby("month", as_index=False)[TARGET]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"count": "tx_count", "mean": "fraud_rate"})
)

print("Monthly fraud rate (train):")
print(train_monthly.tail(6).to_string(index=False))
print("\nMonthly fraud rate (test):")
print(test_monthly.head(6).to_string(index=False))


# 2) Numeric drift: PSI for core numeric fields.
numeric_candidates = ["amt", "city_pop", "lat", "long", "merch_lat", "merch_long", "unix_time"]
numeric_cols = [c for c in numeric_candidates if c in df_train.columns and c in df_test.columns]

numeric_drift_rows = []
for col in numeric_cols:
    psi = population_stability_index(df_train[col], df_test[col], bins=10)
    numeric_drift_rows.append(
        {
            "feature": col,
            "train_mean": float(df_train[col].mean()),
            "test_mean": float(df_test[col].mean()),
            "mean_delta": float(df_test[col].mean() - df_train[col].mean()),
            "train_std": float(df_train[col].std()),
            "test_std": float(df_test[col].std()),
            "psi": psi,
            "psi_risk": psi_risk_band(psi),
        }
    )

numeric_drift = pd.DataFrame(numeric_drift_rows).sort_values("psi", ascending=False)
print("\nNumeric drift summary (sorted by PSI):")
print(numeric_drift.to_string(index=False))


# 3) Categorical drift: compare category share changes.
def categorical_shift(train_df, test_df, col, top_n=10):
    """Return categories with the biggest train-vs-test share change."""
    train_share = train_df[col].astype(str).value_counts(normalize=True)
    test_share = test_df[col].astype(str).value_counts(normalize=True)

    labels = train_share.index.union(test_share.index)
    out = pd.DataFrame(
        {
            "label": labels,
            "train_share": train_share.reindex(labels, fill_value=0.0),
            "test_share": test_share.reindex(labels, fill_value=0.0),
        }
    )
    out["abs_delta"] = (out["test_share"] - out["train_share"]).abs()
    return out.sort_values("abs_delta", ascending=False).head(top_n)


categorical_cols = ["category", "gender", "state", "job"]
categorical_cols = [c for c in categorical_cols if c in df_train.columns and c in df_test.columns]

for col in categorical_cols:
    print(f"\nTop categorical share shifts for '{col}':")
    print(categorical_shift(df_train, df_test, col, top_n=10).to_string(index=False))

print("\nDrift interpretation guide: PSI < 0.10 low, 0.10-0.25 medium, >= 0.25 high.")


Monthly fraud rate (train):
 index   month  tx_count  fraud_rate
    12 2020-01     52202    0.006571
    13 2020-02     47791    0.007031
    14 2020-03     72850    0.006095
    15 2020-04     66892    0.004515
    16 2020-05     74343    0.007089
    17 2020-06     57747    0.005784

Monthly fraud rate (test):
 index   month  tx_count  fraud_rate
     0 2020-06     30058    0.004425
     1 2020-07     85848    0.003739
     2 2020-08     88759    0.004676
     3 2020-09     69533    0.004890
     4 2020-10     69348    0.005537
     5 2020-11     72635    0.004048

Numeric drift summary (sorted by PSI):
   feature    train_mean     test_mean    mean_delta    train_std     test_std       psi psi_risk
 unix_time  1.349244e+09  1.380679e+09  3.143523e+07 1.284128e+07 5.201104e+06 11.512810     high
       amt  7.035104e+01  6.939281e+01 -9.582252e-01 1.603160e+02 1.567459e+02  0.000062      low
  city_pop  8.882444e+04  8.822189e+04 -6.025526e+02 3.019564e+05 3.003909e+05  0.000046    

## Step 4: Fraud Pattern Mining (Understanding)

### Goal
Find interpretable fraud hotspots that can become useful model signals.

### Key concepts for beginners

**What is "lift"?**
Lift measures how much riskier a group is compared to the overall average. A lift of 3.0 means transactions in that group are **3× more likely to be fraud** than a randomly selected transaction.

**Worked numerical example**
- Baseline fraud rate (all transactions): 0.58%
- `shopping_net` category fraud rate: ~1.75%
- Lift = 1.75% ÷ 0.58% ≈ **3.0×**

This means an online shopping transaction is 3× as suspicious as the average. A risk analyst could justify tighter scrutiny for that category.

**Why high-lift AND high-support both matter**
Lift can be misleadingly large for very small groups. Imagine a merchant category with only 10 transactions and 2 frauds — that's a 20% fraud rate and a lift of ~34×. But with 10 data points, we can't trust this signal at all. The `min_tx_count` filter removes groups too small to draw reliable conclusions from. Always check: *is the lift based on enough transactions?*

### What this step checks
- baseline fraud rate
- fraud lift by `category`
- fraud lift by `hour`
- fraud lift by `amt_band`
- interaction hotspots (`category x hour`)

### How to read the output
- `lift > 1`: group is riskier than average
- Use `tx_count` (support) to ignore tiny/noisy groups
- High-lift + high-support patterns are strongest candidates for features/rules.

### Why this matters
Pattern mining explains fraud behavior and helps justify feature choices in the report.

### What to report
- top high-lift categories/hours/amount bands
- top interaction hotspots
- how these patterns inform feature engineering


In [5]:
# ---------------------------------------------------------------------------
# Step 4: Fraud pattern mining
# ---------------------------------------------------------------------------
# Goal: discover interpretable fraud hotspots for feature design.

TARGET = "is_fraud"


def fraud_lift_table(df, group_col, min_tx_count=1000):
    """Compute fraud lift per group relative to the global baseline rate."""
    baseline = df[TARGET].mean()

    # .agg() with keyword arguments creates named result columns in one step:
    # tx_count="count"  → total transactions in this group
    # fraud_count="sum" → sum of is_fraud column (1s = fraud, 0s = not fraud)
    # fraud_rate="mean" → mean of is_fraud = fraction that are fraud
    summary = (
        df.groupby(group_col, dropna=False)[TARGET]
        .agg(tx_count="count", fraud_count="sum", fraud_rate="mean")
        .reset_index()
    )

    # min_tx_count is a "support filter": groups with too few transactions have
    # noisy fraud rates that can mislead feature engineering decisions.
    # Example: a group of 5 transactions with 1 fraud is 20% fraud — but that
    # is almost certainly statistical noise, not a real signal.
    summary = summary[summary["tx_count"] >= min_tx_count].copy()
    summary["lift_vs_baseline"] = summary["fraud_rate"] / baseline
    summary = summary.sort_values("lift_vs_baseline", ascending=False)

    return summary, baseline


# Use train split only for exploratory pattern mining.
# This avoids leaking test information into feature decisions.
df_pattern = df_train.copy()
df_pattern["hour"] = pd.to_datetime(df_pattern["trans_date_trans_time"]).dt.hour

# 1) Baseline rate used as denominator for lift.
baseline_rate = df_pattern[TARGET].mean()
print(f"Baseline fraud rate (train): {baseline_rate:.4%}")

# 2) Category-level lift.
cat_lift, _ = fraud_lift_table(df_pattern, "category", min_tx_count=10000)
print("\nTop category fraud lift (train):")
print(cat_lift[["category", "tx_count", "fraud_count", "fraud_rate", "lift_vs_baseline"]].head(10).to_string(index=False))

# 3) Hour-level lift.
hour_lift, _ = fraud_lift_table(df_pattern, "hour", min_tx_count=1000)
print("\nTop hour-of-day fraud lift (train):")
print(hour_lift[["hour", "tx_count", "fraud_count", "fraud_rate", "lift_vs_baseline"]].head(10).to_string(index=False))

# 4) pd.cut() divides a continuous variable (transaction amount) into labelled buckets.
# bins defines the edges: [0, 10, 25, ..., inf]
# labels assigns a readable name to each bucket: "0-10", "10-25", etc.
# include_lowest=True ensures the lowest edge (0) is included in the first bin.
amount_bins = [0, 10, 25, 50, 100, 200, 500, 1000, np.inf]
amount_labels = ["0-10", "10-25", "25-50", "50-100", "100-200", "200-500", "500-1000", "1000+"]
df_pattern["amt_band"] = pd.cut(df_pattern["amt"], bins=amount_bins, labels=amount_labels, include_lowest=True)

amt_lift, _ = fraud_lift_table(df_pattern, "amt_band", min_tx_count=1000)
print("\nTop amount-band fraud lift (train):")
print(amt_lift[["amt_band", "tx_count", "fraud_count", "fraud_rate", "lift_vs_baseline"]].to_string(index=False))

# 5) Interaction hotspots: high-risk combinations of category and hour.
hotspots = (
    df_pattern.groupby(["category", "hour"], dropna=False)[TARGET]
    .agg(tx_count="count", fraud_count="sum", fraud_rate="mean")
    .reset_index()
)
hotspots = hotspots[hotspots["tx_count"] >= 500].copy()
hotspots["lift_vs_baseline"] = hotspots["fraud_rate"] / baseline_rate
hotspots = hotspots.sort_values(["lift_vs_baseline", "fraud_count"], ascending=[False, False])

print("\nTop interaction hotspots (category x hour, train):")
print(hotspots[["category", "hour", "tx_count", "fraud_count", "fraud_rate", "lift_vs_baseline"]].head(15).to_string(index=False))

print("\nInterpretation note: prioritize high-lift groups with enough support for feature engineering and threshold policy.")


Baseline fraud rate (train): 0.5789%

Top category fraud lift (train):
     category  tx_count  fraud_count  fraud_rate  lift_vs_baseline
 shopping_net     97543         1713    0.017561          3.033778
     misc_net     63287          915    0.014458          2.497636
  grocery_pos    123638         1743    0.014098          2.435387
 shopping_pos    116672          843    0.007225          1.248198
gas_transport    131659          618    0.004694          0.810887
     misc_pos     79655          250    0.003139          0.542188
  grocery_net     45452          134    0.002948          0.509301
       travel     40507          116    0.002864          0.494710
entertainment     94014          233    0.002478          0.428140
personal_care     90758          220    0.002424          0.418755

Top hour-of-day fraud lift (train):
 hour  tx_count  fraud_count  fraud_rate  lift_vs_baseline
   22     66982         1931    0.028829          4.980200
   23     67104         1904    0.028

## Step 5: Leakage and Proxy-Risk Analysis

### Goal
Identify raw columns that may create unrealistic model performance or fairness risks.

### Key concepts for beginners

**What is data leakage?**
Leakage happens when the model learns from information it wouldn't have access to in production. A classic analogy: *giving a student the exam answers during a practice test makes them look smarter than they are* — they memorize the answers, not the concepts. When deployed in the real world, they fail. In fraud detection, a column like `trans_num` (a unique transaction ID) perfectly identifies each row — a model trained on it would score 100% on training data but learn nothing useful for future transactions.

**Why `cc_num` overlap is a leakage risk**
There are 908 credit card numbers (`cc_num`) that appear in both train and test. This means the model could learn patterns tied to specific cards: "card #4532... was flagged 5 times before." In production, new cardholders with unseen card numbers would be systematically under-scored. The card number is a direct identifier — it tells you *who*, not *why* — so it must be dropped.

**Why `trans_date_trans_time` is "engineer then drop"**
The raw timestamp is dangerous if used directly: it encodes time ordering (test transactions always come after train ones), which a model could exploit as a shortcut rather than learning true fraud signals. However, the *signals within* the timestamp — hour of day, day of week, month — are safe and useful features that don't reveal the train/test split. The strategy: extract those signals, then drop the raw column.

### What this step checks
- direct identifiers (`trans_num`, `cc_num`, names, street)
- raw time fields that can leak sequence artifacts
- personal/location proxy features
- near-unique high-cardinality fields

### How to read the output
- `drop_now`: do not use raw feature in final model
- `engineer_then_drop_raw`: create safe derived features, then remove raw column
- `keep_with_monitor`: allowed, but track drift/performance impact
- `keep`: low-risk feature

### Why this matters
Leakage-safe features are required for honest evaluation and stable production behavior.

### What to report
- feature risk table (risk + reason + recommendation)
- final keep/drop/watch decisions
- identifier overlap audit notes (if available)


In [6]:
# ---------------------------------------------------------------------------
# Step 5: Leakage and proxy-risk analysis
# ---------------------------------------------------------------------------
# Goal: identify columns that can create unrealistic model performance.

TARGET = "is_fraud"

# Risk dictionaries based on domain knowledge.
DIRECT_IDENTIFIER_COLS = {"trans_num", "cc_num", "first", "last", "street"}
TEMPORAL_RAW_COLS = {"trans_date_trans_time", "unix_time"}
PERSONAL_PROXY_COLS = {"dob", "zip", "city", "state", "job"}


def classify_feature_risk(col_name, unique_ratio):
    """Assign risk level + recommended action for one raw feature."""
    # Step-by-step decision logic:
    # 1. Direct identifiers (card numbers, names, IDs) are the highest risk:
    #    they uniquely identify individual rows and would cause the model to
    #    memorize training examples rather than learn generalizable patterns.
    if col_name in DIRECT_IDENTIFIER_COLS:
        return "high", "drop_now", "direct_identifier"
    # 2. Raw timestamps encode time ordering — the model could learn
    #    "test transactions always have higher unix_time" rather than real fraud
    #    signals. Extract safe signals (hour, day) first, then drop the raw column.
    if col_name in TEMPORAL_RAW_COLS:
        return "high", "engineer_then_drop_raw", "raw_time_granularity"
    # 3. Personal/location fields are medium risk: not direct identifiers,
    #    but could introduce demographic proxies or geographic memorization.
    if col_name in PERSONAL_PROXY_COLS:
        return "medium", "keep_with_monitor", "personal_or_location_proxy"
    # 4. Near-unique columns (almost every value is distinct) act like IDs:
    #    unique_ratio close to 1.0 means the model would overfit to individual rows.
    if unique_ratio >= 0.95:
        return "medium", "keep_with_monitor", "near_unique_high_cardinality"
    # 5. Everything else passes — low risk, safe to use as a model feature.
    return "low", "keep", "no_major_risk"


feature_rows = []
feature_cols = [c for c in df_train.columns if c != TARGET]

for col in feature_cols:
    s = df_train[col]
    non_null = s.notna().sum()
    nunique = int(s.nunique(dropna=True))
    # unique_ratio = fraction of non-null values that are unique.
    # Near 1.0 means almost every row has a different value → behaves like an ID.
    # Near 0.0 means the column has very few distinct values → categorical-like.
    unique_ratio = (nunique / non_null) if non_null else 0.0

    risk_level, recommendation, reason = classify_feature_risk(col, unique_ratio)

    feature_rows.append(
        {
            "feature": col,
            "dtype": str(s.dtype),
            "non_null": int(non_null),
            "nunique": nunique,
            "unique_ratio": float(unique_ratio),
            "risk_level": risk_level,
            "reason": reason,
            "recommendation": recommendation,
        }
    )

risk_df = pd.DataFrame(feature_rows)

# Sort by risk first so report readers see the most important issues first.
risk_order = {"high": 0, "medium": 1, "low": 2}
risk_df["_risk_order"] = risk_df["risk_level"].map(risk_order)
risk_df = risk_df.sort_values(["_risk_order", "unique_ratio"], ascending=[True, False]).drop(columns=["_risk_order"])

print("Feature risk assessment:")
print(risk_df.to_string(index=False))


# Build action buckets for implementation planning.
drop_now = risk_df.loc[risk_df["recommendation"] == "drop_now", "feature"].tolist()
engineer_then_drop_raw = risk_df.loc[risk_df["recommendation"] == "engineer_then_drop_raw", "feature"].tolist()
keep_with_monitor = risk_df.loc[risk_df["recommendation"] == "keep_with_monitor", "feature"].tolist()
keep = risk_df.loc[risk_df["recommendation"] == "keep", "feature"].tolist()

print("\nRecommended feature actions:")
print(f"  drop_now               : {drop_now}")
print(f"  engineer_then_drop_raw : {engineer_then_drop_raw}")
print(f"  keep_with_monitor      : {keep_with_monitor}")
print(f"  keep                   : {keep}")

# Optional overlap checks for audit transparency.
# These reveal whether the same individual identifiers appear in both splits,
# which would allow the model to "recognize" known cards or transactions.
if "trans_num" in df_train.columns and "trans_num" in df_test.columns:
    overlap_trans_num = len(set(df_train["trans_num"]).intersection(set(df_test["trans_num"])))
    print(f"\nIdentifier overlap check: trans_num overlap train/test = {overlap_trans_num}")

if "cc_num" in df_train.columns and "cc_num" in df_test.columns:
    overlap_cc_num = len(set(df_train["cc_num"]).intersection(set(df_test["cc_num"])))
    print(f"Identifier overlap check: cc_num overlap train/test = {overlap_cc_num}")

print("\nGuideline: high-risk columns should not enter the final model as raw features.")


Feature risk assessment:
              feature   dtype  non_null  nunique  unique_ratio risk_level                       reason         recommendation
            trans_num  object   1296675  1296675      1.000000       high            direct_identifier               drop_now
            unix_time   int64   1296675  1274823      0.983148       high         raw_time_granularity engineer_then_drop_raw
trans_date_trans_time  object   1296675  1274791      0.983123       high         raw_time_granularity engineer_then_drop_raw
               cc_num   int64   1296675      983      0.000758       high            direct_identifier               drop_now
               street  object   1296675      983      0.000758       high            direct_identifier               drop_now
                 last  object   1296675      481      0.000371       high            direct_identifier               drop_now
                first  object   1296675      352      0.000271       high            direct_i

## Step 6: Feature Strategy Decision (Precision-First)

### Goal
Convert analysis findings (Steps 1–5) into a final, implementation-ready feature policy.

### Key concepts for beginners

**The "fit on train only" rule**
The most common beginner mistake in preprocessing is fitting transformers on the full dataset before splitting — or after the split but on combined data. Your preprocessing must not *peek* at test data. If you compute the mean of a column to fill missing values, or learn the vocabulary of categories, do it using *only* training data. Fitting on combined train+test is a subtle form of leakage: the scaler "knows" about test-set values, which slightly inflates apparent model performance.

**Preprocessing decision table**

| Column type | Action | Reason |
|-------------|--------|--------|
| Direct identifier (cc_num, trans_num) | Drop immediately | Memorizes rows, not patterns |
| Raw timestamp (trans_date_trans_time) | Engineer → drop raw | Extract hour/day, remove original |
| Geo coordinates (lat/long) | Engineer → drop raw | Combine into distance_km |
| Text categories (merchant, category) | LabelEncode (train only) | Convert to integers |
| Numeric features (amt, city_pop) | StandardScale (train only) | Equalize feature magnitudes |

**Short glossary**
- **LabelEncoder**: learns a mapping from category names → integers. `fit()` learns the vocabulary; `transform()` applies it. Only call `fit()` on training data.
- **StandardScaler**: rescales each numeric column to mean=0, std=1. Call `fit_transform()` on train; `transform()` only on test.
- **Feature engineering**: creating new, more informative columns (like `distance_km`, `hour`) from raw columns (like lat/long, timestamps) that better capture the underlying pattern.

### What this step decides
- which raw columns are dropped immediately
- which raw columns are used to create derived features, then dropped
- which columns are kept as model inputs
- categorical encoding and scaling policy

### How to read the output
- The feature decision table is the source of truth for preprocessing.
- `engineer_then_drop_raw` means: keep temporarily only to build new features.
- `keep_raw` means: column can remain in the model input after preprocessing.

### Why this matters
A clear feature policy avoids leakage, keeps modeling reproducible, and makes implementation easier for other developers.

### What to report
- final keep/drop/engineer decisions per feature
- final derived feature list
- categorical feature list
- preprocessing policy (encoding, scaling, imbalance handling)


In [7]:
# ---------------------------------------------------------------------------
# Step 6: Feature strategy decision (precision-first)
# ---------------------------------------------------------------------------
# Goal: create one clear feature policy that downstream cells can reuse.

TARGET = "is_fraud"

# 1) Final decisions from previous analysis steps.
# DROP_NOW_COLS: direct identifiers and high-proxy-risk fields.
#   These columns must not enter the model as raw inputs — they either
#   uniquely identify individual rows (leakage) or encode protected attributes.
DROP_NOW_COLS = [
    "cc_num", "first", "last", "street", "trans_num", "zip", "city", "state"
]

# DERIVED_FEATURE_MAP: maps each raw column → list of safe features it produces.
#   After engineering, the raw column is dropped. The derived features remain.
#   Example: "trans_date_trans_time" → ["hour", "day_of_week", "month"]
#   "unix_time" maps to [] because no safe derived signal is extracted from it.
DERIVED_FEATURE_MAP = {
    "trans_date_trans_time": ["hour", "day_of_week", "month"],
    "dob": ["age"],
    "lat": ["distance_km"],
    "long": ["distance_km"],
    "merch_lat": ["distance_km"],
    "merch_long": ["distance_km"],
    "unix_time": []
}

# KEEP_RAW_COLS: columns safe to use directly as model features after encoding/scaling.
KEEP_RAW_COLS = ["merchant", "category", "amt", "gender", "city_pop", "job"]

# 2) Downstream preprocessing policy.
# CATEGORICAL_FEATURES: will be LabelEncoded (fit on train only).
CATEGORICAL_FEATURES = ["merchant", "category", "gender", "job"]
# NUMERIC_FEATURES_AFTER_ENGINEERING: will be StandardScaled (fit on train only).
NUMERIC_FEATURES_AFTER_ENGINEERING = ["amt", "city_pop", "hour", "day_of_week", "month", "age", "distance_km"]
# AGE_REFERENCE_DATE: date used to compute customer age from date-of-birth.
#   Using the last training date keeps age consistent and leakage-free.
AGE_REFERENCE_DATE = pd.Timestamp("2020-06-21")

# sorted(set(...)) deduplicates and sorts alphabetically for deterministic output.
# Both DROP_NOW_COLS and DERIVED_FEATURE_MAP keys must be removed after engineering.
RAW_DROP_AFTER_ENGINEERING = sorted(set(DROP_NOW_COLS + list(DERIVED_FEATURE_MAP.keys())))

PREPROCESSING_POLICY = {
    "categorical_encoding": "LabelEncoder (fit on train only, unseen in test -> -1)",
    "scaling": "StandardScaler (fit on train only)",
    "imbalance_default": "class_weight",
    "objective": "high_precision"
}

# 3) Build a transparent per-feature decision table for reporting.
feature_rows = []
for col in df_train.columns:
    if col == TARGET:
        action = "target"
        reason = "prediction_target"
        derived_outputs = []
    elif col in DROP_NOW_COLS:
        action = "drop_now"
        reason = "identifier_or_high_proxy_risk"
        derived_outputs = []
    elif col in DERIVED_FEATURE_MAP:
        action = "engineer_then_drop_raw"
        reason = "convert_raw_signal_into_safer_feature"
        derived_outputs = DERIVED_FEATURE_MAP[col]
    elif col in KEEP_RAW_COLS:
        action = "keep_raw"
        reason = "useful_predictive_signal"
        derived_outputs = []
    else:
        action = "review_needed"
        reason = "not_in_explicit_policy"
        derived_outputs = []

    feature_rows.append(
        {
            "feature": col,
            "action": action,
            "reason": reason,
            "derived_outputs": ", ".join(derived_outputs)
        }
    )

feature_strategy_df = pd.DataFrame(feature_rows).sort_values(["action", "feature"]).reset_index(drop=True)

# 4) Sanity checks for student-friendly debugging.
missing_keep = [c for c in KEEP_RAW_COLS if c not in df_train.columns]
missing_derived = [c for c in DERIVED_FEATURE_MAP.keys() if c not in df_train.columns]
assert not missing_keep, f"Missing keep columns in train data: {missing_keep}"
assert not missing_derived, f"Missing raw columns for derived features: {missing_derived}"

print("Feature strategy table:")
print(feature_strategy_df.to_string(index=False))

print("\nFinal policy summary:")
print(f"  drop_now_cols                : {DROP_NOW_COLS}")
print(f"  engineer_then_drop_raw_cols  : {list(DERIVED_FEATURE_MAP.keys())}")
print(f"  keep_raw_cols                : {KEEP_RAW_COLS}")
print(f"  derived_features             : {sorted(set(sum(DERIVED_FEATURE_MAP.values(), [])))}")
print(f"  categorical_features         : {CATEGORICAL_FEATURES}")
print(f"  numeric_features             : {NUMERIC_FEATURES_AFTER_ENGINEERING}")
print(f"  preprocessing_policy         : {PREPROCESSING_POLICY}")


Feature strategy table:
              feature                 action                                reason          derived_outputs
               cc_num               drop_now         identifier_or_high_proxy_risk                         
                 city               drop_now         identifier_or_high_proxy_risk                         
                first               drop_now         identifier_or_high_proxy_risk                         
                 last               drop_now         identifier_or_high_proxy_risk                         
                state               drop_now         identifier_or_high_proxy_risk                         
               street               drop_now         identifier_or_high_proxy_risk                         
            trans_num               drop_now         identifier_or_high_proxy_risk                         
                  zip               drop_now         identifier_or_high_proxy_risk                         
    

In [8]:
# ---------------------------------------------------------------------------
# Step 7: Feature engineering (Part A) — spatial, temporal, and encoded features
# ---------------------------------------------------------------------------
# Goal: transform raw transaction data into model-friendly predictive features.


def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized Haversine distance in kilometers."""
    # The Haversine formula treats Earth as a sphere (radius R ≈ 6371 km).
    # It converts lat/lon degrees to radians, computes the central angle
    # between the two points using arc-trigonometry, then multiplies by R
    # to get the shortest surface path (the "great-circle" distance).
    # np.radians() converts degrees → radians (required by trig functions).
    # arctan2 is used instead of arcsin for numerical stability near the poles.
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


def engineer_features(df):
    """Create engineered features and return a transformed copy."""
    df = df.copy()

    # Time-based behavior features from raw transaction timestamp.
    ts = pd.to_datetime(df["trans_date_trans_time"])
    df["hour"] = ts.dt.hour
    df["day_of_week"] = ts.dt.dayofweek  # 0=Monday, 6=Sunday
    df["month"] = ts.dt.month

    # Customer age at reference date. Step 6 can override this date if needed.
    reference_date = globals().get("AGE_REFERENCE_DATE", pd.Timestamp("2020-06-21"))
    df["age"] = ((reference_date - pd.to_datetime(df["dob"])).dt.days / 365.25).round(1)

    # Cardholder-to-merchant geographic distance.
    df["distance_km"] = haversine_km(
        df["lat"], df["long"], df["merch_lat"], df["merch_long"]
    )

    # Prefer Step 6 drop policy; fallback keeps this cell runnable standalone.
    default_cols_to_drop = [
        "trans_date_trans_time", "cc_num", "first", "last",
        "street", "city", "state", "zip",
        "lat", "long", "merch_lat", "merch_long",
        "trans_num", "unix_time", "dob",
    ]
    cols_to_drop = globals().get("RAW_DROP_AFTER_ENGINEERING", default_cols_to_drop)
    return df.drop(columns=cols_to_drop)


# Apply identical feature logic to both splits.
df_train_eng = engineer_features(df_train)
df_test_eng = engineer_features(df_test)

print(f"Train after engineering: {df_train_eng.shape}")
print(f"Columns: {df_train_eng.columns.tolist()}")
print(df_train_eng.dtypes)

Train after engineering: (1296675, 12)
Columns: ['merchant', 'category', 'amt', 'gender', 'city_pop', 'job', 'is_fraud', 'hour', 'day_of_week', 'month', 'age', 'distance_km']
merchant        object
category        object
amt            float64
gender          object
city_pop         int64
job             object
is_fraud         int64
hour             int32
day_of_week      int32
month            int32
age            float64
distance_km    float64
dtype: object


In [9]:
# ---------------------------------------------------------------------------
# Encoding, scaling, and NumPy export
# ---------------------------------------------------------------------------
# Goal: convert engineered DataFrames into model-ready matrices.

# Prefer Step 6 policy values; fallback keeps this cell runnable standalone.
CATEGORICAL_COLS = globals().get("CATEGORICAL_FEATURES", ["merchant", "category", "gender", "job"])
TARGET = "is_fraud"

# LabelEncoder workflow:
#   le.fit(train_values)  → learns the vocabulary: {"gas_transport": 0, "grocery_net": 1, ...}
#   le.transform(values)  → applies the mapping: "grocery_net" → 1
# CRITICAL: fit on TRAIN only. Fitting on test would reveal test-set category
# frequencies to the preprocessing step — a subtle form of leakage.
encoders = {}
df_train_enc = df_train_eng.copy()
df_test_enc = df_test_eng.copy()

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    le.fit(df_train_enc[col])
    encoders[col] = le

    # Normal transform on training categories.
    df_train_enc[col] = le.transform(df_train_enc[col])

    # For test data, build an explicit dict from the trained encoder:
    # dict(zip(le.classes_, le.transform(le.classes_))) → {"gas_transport": 0, ...}
    # .map(mapping) replaces each string with its integer code.
    # .fillna(-1) handles test categories not seen in training — mapped to -1
    # as a safe sentinel value. Without this, le.transform() would crash on
    # any new label that wasn't in the training vocabulary.
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    df_test_enc[col] = df_test_enc[col].map(mapping).fillna(-1).astype(int)

    unseen = (df_test_enc[col] == -1).sum()
    print(f"  {col:12s}: {len(le.classes_)} train classes | {unseen} unseen -> -1")

# Split predictors and target.
FEATURE_COLS = [c for c in df_train_enc.columns if c != TARGET]

X_train = df_train_enc[FEATURE_COLS]
y_train = df_train_enc[TARGET]
X_test = df_test_enc[FEATURE_COLS]
y_test = df_test_enc[TARGET]

print(f"\nX_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   |  y_test : {y_test.shape}")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

# Standardize numeric scale.
# scaler.fit_transform(X_train): LEARNS mean/std from train, then scales train data.
# scaler.transform(X_test):      Uses ALREADY-LEARNED mean/std to scale test data.
# Never call fit_transform on test — that would compute test statistics and
# subtly shift the scaling, breaking the "train-only fit" contract.
scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train)
X_test_np = scaler.transform(X_test)

# to_numpy(dtype=np.int32): converts a pandas Series to a plain NumPy array
# with 32-bit integer type. Required because TensorFlow and some sklearn
# functions expect NumPy arrays rather than pandas Series objects.
y_train_np = y_train.to_numpy(dtype=np.int32)
y_test_np = y_test.to_numpy(dtype=np.int32)

# Final audit of model-ready arrays.
print("\n=== Final NumPy arrays ===")
print(f"X_train_np : {X_train_np.shape}  dtype={X_train_np.dtype}")
print(f"X_test_np  : {X_test_np.shape}   dtype={X_test_np.dtype}")
print(f"y_train_np : {y_train_np.shape}  fraud={y_train_np.sum()} ({y_train_np.mean():.4%})")
print(f"y_test_np  : {y_test_np.shape}   fraud={y_test_np.sum()} ({y_test_np.mean():.4%})")

# Keep unscaled DataFrames for EDA/debugging if needed.


  merchant    : 693 train classes | 0 unseen -> -1
  category    : 14 train classes | 0 unseen -> -1
  gender      : 2 train classes | 0 unseen -> -1
  job         : 494 train classes | 30 unseen -> -1

X_train: (1296675, 11)  |  y_train: (1296675,)
X_test : (555719, 11)   |  y_test : (555719,)
Features (11): ['merchant', 'category', 'amt', 'gender', 'city_pop', 'job', 'hour', 'day_of_week', 'month', 'age', 'distance_km']

=== Final NumPy arrays ===
X_train_np : (1296675, 11)  dtype=float64
X_test_np  : (555719, 11)   dtype=float64
y_train_np : (1296675,)  fraud=7506 (0.5789%)
y_test_np  : (555719,)   fraud=2145 (0.3860%)


## Step 7: Baseline Modeling (Precision-First)

### Goal
Train simple, reproducible baseline models and compare them with fraud-appropriate metrics.

### Key concepts for beginners

**Logistic Regression — plain English**
Logistic Regression finds a *straight line* (or hyperplane in multi-dimensional space) that best separates fraud from non-fraud transactions. It works by computing a weighted sum of all input features and passing the result through a sigmoid function to produce a probability between 0 and 1. Simple, fast, and interpretable — if the coefficient for `distance_km` is large and positive, greater distance strongly predicts fraud.

**Random Forest — plain English**
A Random Forest builds hundreds of *decision trees*, each trained on a random subset of the data and features. Each tree independently votes on whether a transaction is fraud; the final prediction averages all votes. Because each tree sees slightly different data, the ensemble is much more robust than any single tree and naturally captures non-linear interactions (e.g., "high amount AND late night AND online category → high fraud risk").

**`class_weight="balanced"` — why it matters**
With 171:1 imbalance, a model without class weighting will naturally minimize errors on the majority class (non-fraud), because there are so many more of them. `class_weight="balanced"` re-weights the loss function so each fraud mistake is treated as ~171× more costly than a non-fraud mistake. This pushes the model to pay real attention to the rare class.

**`predict_proba` vs `predict`**
- `model.predict_proba(X)[:, 1]`: returns a continuous *probability* between 0 and 1 for each transaction being fraud. Column 0 = P(not fraud), Column 1 = P(fraud). We use column 1.
- `model.predict(X)`: returns a hard 0/1 decision using the default threshold of 0.5. We compute this ourselves from `predict_proba` so we can adjust the threshold freely in Step 8.

### Models currently involved
- `log_reg_balanced` (`LogisticRegression`): linear baseline with `class_weight="balanced"`
- `rf_balanced` (`RandomForestClassifier`): non-linear tree ensemble with `class_weight="balanced_subsample"`

### Model schematics (planned implementation flow)
`Raw data` -> `Step 6 feature policy` -> `Feature engineering` -> `Encoding + scaling` -> `Model` -> `Fraud probability score` -> `Threshold (Step 8)` -> `Fraud alert / no alert`

### Why these two baselines first
- Logistic Regression gives a simple, interpretable benchmark.
- Random Forest captures non-linear patterns and interactions.
- Together, they provide a strong minimum benchmark before trying more complex models.

### What this step does
- trains class-weighted baseline models
- evaluates ranking quality (`ROC-AUC`, `PR-AUC`)
- evaluates alert quality at default threshold (`precision`, `recall`, alert rate)
- produces a comparison table for model selection

### How to read the output
- Higher `PR-AUC` is usually more important than accuracy in fraud tasks.
- Higher `precision_at_0_5` means fewer false alerts.
- `alert_rate_at_0_5` shows operational workload impact.

### Why this matters
This creates a transparent starting point before threshold tuning (Step 8) and final recommendation (Step 9).

### What to report
- baseline model comparison table
- best candidate by `PR-AUC`
- tradeoff notes for precision vs recall
- model schematic used for implementation


In [10]:
# ---------------------------------------------------------------------------
# Step 7: Baseline modeling (precision-first)
# ---------------------------------------------------------------------------
# Goal: train simple baseline models and compare them with fraud-focused metrics.
#
# Models used in this step:
# 1) Logistic Regression (class-weighted)
#    - Structure: linear decision boundary in transformed feature space.
#    - Role: interpretable baseline and sanity check.
#
# 2) Random Forest (class-weighted)
#    - Structure: ensemble of decision trees with majority/probability voting.
#    - Role: captures non-linear interactions and serves as stronger baseline.
#
# Planned pipeline schematic:
# Raw/engineered features -> model.predict_proba(...) -> fraud score
# fraud score + threshold (Step 8) -> alert decision

from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score

# RANDOM_STATE=42 is a convention (the number itself is arbitrary).
# Setting a fixed seed makes the random number generator deterministic:
# every run produces the same tree structures and results. Without it,
# model performance numbers would change slightly between runs.
RANDOM_STATE = 42

# Baseline model set (class-weighted for severe class imbalance).
baseline_models = {
    "log_reg_balanced": LogisticRegression(
        class_weight="balanced",   # re-weights loss: fraud errors cost ~171x more
        max_iter=1000,             # allow more iterations if solver hasn't converged
        random_state=RANDOM_STATE
    ),
    "rf_balanced": RandomForestClassifier(
        n_estimators=300,                    # build 300 decision trees
        class_weight="balanced_subsample",   # re-balance weights per bootstrap sample
        random_state=RANDOM_STATE,
        n_jobs=-1                            # use all available CPU cores in parallel
    ),
}

# Student-friendly model summary for report/debug output.
MODEL_SUMMARY = {
    "log_reg_balanced": {
        "family": "linear model",
        "core_structure": "single linear decision function",
        "strength": "interpretable and fast",
    },
    "rf_balanced": {
        "family": "tree ensemble",
        "core_structure": "many decision trees aggregated",
        "strength": "captures non-linear patterns",
    },
}

print("Models involved in Step 7:")
for name, info in MODEL_SUMMARY.items():
    print(f"  - {name}: {info['family']} | structure={info['core_structure']} | strength={info['strength']}")


def evaluate_binary_model(name, model, X_tr, y_tr, X_te, y_te, threshold=0.5):
    """Fit one model and return fraud-relevant evaluation metrics."""
    # model.fit(X_tr, y_tr): trains the model by adjusting its internal parameters
    # to minimize prediction errors on the training data. After this call, the
    # model "knows" the relationship between features and fraud labels — at least
    # for the training examples it was shown.
    model.fit(X_tr, y_tr)

    # predict_proba returns a 2D array: each row = [P(not fraud), P(fraud)].
    # [:, 1] selects column index 1 = P(fraud) for every test transaction.
    # We use this continuous score for threshold-free ranking metrics (ROC-AUC, PR-AUC).
    y_score = model.predict_proba(X_te)[:, 1]
    # Convert probability to a hard 0/1 prediction using the threshold.
    # (y_score >= threshold) → boolean array; .astype(int) converts True→1, False→0.
    y_pred = (y_score >= threshold).astype(int)

    roc_auc = roc_auc_score(y_te, y_score)
    # average_precision_score = area under the precision-recall curve (PR-AUC).
    # This is our primary metric: it summarizes model quality across all thresholds
    # and is far more sensitive to rare-class performance than ROC-AUC.
    pr_auc = average_precision_score(y_te, y_score)
    # precision: of all flagged transactions, how many are truly fraud?
    precision = precision_score(y_te, y_pred, zero_division=0)
    # recall: of all true fraud transactions, how many did we catch?
    recall = recall_score(y_te, y_pred, zero_division=0)
    f1 = f1_score(y_te, y_pred, zero_division=0)

    alert_count = int(y_pred.sum())
    alert_rate = float(alert_count / len(y_pred))

    metrics = {
        "model": name,
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "precision_at_0_5": float(precision),
        "recall_at_0_5": float(recall),
        "f1_at_0_5": float(f1),
        "alert_count_at_0_5": alert_count,
        "alert_rate_at_0_5": alert_rate,
    }

    artifacts = {
        "model": model,
        "y_score_test": y_score,
        "y_pred_test_at_0_5": y_pred,
    }

    return metrics, artifacts


baseline_metrics = []
baseline_artifacts = {}

for model_name, model in baseline_models.items():
    print(f"\nTraining baseline model: {model_name}")

    metrics, artifacts = evaluate_binary_model(
        name=model_name,
        model=model,
        X_tr=X_train_np,
        y_tr=y_train_np,
        X_te=X_test_np,
        y_te=y_test_np,
        threshold=0.5,
    )

    baseline_metrics.append(metrics)
    baseline_artifacts[model_name] = artifacts


# Create a clean comparison table for report use.
baseline_results_df = pd.DataFrame(baseline_metrics).sort_values("pr_auc", ascending=False).reset_index(drop=True)

print("\nBaseline model comparison (sorted by PR-AUC):")
print(baseline_results_df.to_string(index=False))

# We select the model with the highest PR-AUC (not ROC-AUC or accuracy) because:
# PR-AUC directly measures performance on the rare positive class (fraud).
# ROC-AUC is inflated by the model's excellent performance on the 99.4% majority.
# Accuracy is meaningless here — see the "lazy model" in Step 2.
best_model_name = baseline_results_df.loc[0, "model"]
best_model = baseline_artifacts[best_model_name]["model"]
best_model_test_scores = baseline_artifacts[best_model_name]["y_score_test"]

print(f"\nSelected baseline candidate for Step 8: {best_model_name}")
print("Selection rule: highest PR-AUC, then inspect precision/alert-rate tradeoff.")


Models involved in Step 7:
  - log_reg_balanced: linear model | structure=single linear decision function | strength=interpretable and fast
  - rf_balanced: tree ensemble | structure=many decision trees aggregated | strength=captures non-linear patterns

Training baseline model: log_reg_balanced

Training baseline model: rf_balanced

Baseline model comparison (sorted by PR-AUC):
           model  roc_auc   pr_auc  precision_at_0_5  recall_at_0_5  f1_at_0_5  alert_count_at_0_5  alert_rate_at_0_5
     rf_balanced 0.987662 0.883456          0.957617       0.726807   0.826398                1628           0.002930
log_reg_balanced 0.851319 0.145177          0.075841       0.741259   0.137603               20965           0.037726

Selected baseline candidate for Step 8: rf_balanced
Selection rule: highest PR-AUC, then inspect precision/alert-rate tradeoff.


## Step 8: SMOTE Oversampling

### What is SMOTE?

`class_weight="balanced"` changes *how much each error is penalised* during training — it tells the model "a missed fraud costs 171× more than a missed legitimate transaction". But the Random Forest still **sees** 171 non-fraud examples for every 1 fraud example when building each tree. Most splits will still carve out majority-class territory.

**SMOTE** (Synthetic Minority Oversampling TEchnique) takes a different approach: instead of just repeating the same few fraud examples, it *creates new realistic-looking ones* by blending existing fraud cases together. Concretely, for each real fraud transaction, SMOTE finds its nearest fraud neighbours and generates synthetic points along the line segments between them. The model trains on a genuinely balanced dataset.

> **Rule**: SMOTE is applied to the **training set only**. The test set must remain untouched — it represents the real-world distribution the model will face in production.

### What to expect
- **Recall** should increase (the model sees more fraud signal during training)
- **Precision** may drop slightly — SMOTE trades some false-alarm reduction for better fraud capture
- **Alert count** will likely rise — more flagged transactions, with a higher share being real fraud

### Preview: comparison table
After training, you will see a side-by-side table:

| model | pr_auc | recall_at_0_5 | precision_at_0_5 | alert_count_at_0_5 |
|-------|--------|---------------|------------------|--------------------|
| rf_balanced | baseline | ... | ... | ... |
| rf_smote | improved? | ↑ recall | ↓ precision? | ↑ alerts? |

In [11]:
from imblearn.over_sampling import SMOTE

# SMOTE must only see training data — it would be cheating to let it
# generate synthetic examples based on test labels or feature distributions.
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train_np, y_train_np)
# fit_resample() does two things at once: it learns the fraud neighbourhood
# structure (fit) and creates synthetic fraud rows until classes are balanced
# (resample). X_train_sm has the same columns as X_train_np but more rows.

# We keep balanced_subsample instead of switching to balanced here because
# SMOTE already handles the class ratio — balanced_subsample re-weights within
# each bootstrap sample as an extra safety net, not the primary mechanism.
rf_smote = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1  # use all CPU cores; RF trees are independent so this is safe
)
smote_metrics, smote_artifacts = evaluate_binary_model(
    "rf_smote", rf_smote,
    X_train_sm, y_train_sm,  # train on the SMOTE-augmented set
    X_test_np, y_test_np     # always evaluate on the original, untouched test set
)

# iloc[0] picks the best baseline model row (baseline_results_df is sorted by
# PR-AUC descending, so row 0 is always the strongest baseline).
comparison_df = pd.DataFrame([baseline_results_df.iloc[0].to_dict(), smote_metrics])
print(comparison_df[["model", "pr_auc", "recall_at_0_5", "precision_at_0_5", "alert_count_at_0_5"]])


def print_improvement(df):
    base, new = df.iloc[0], df.iloc[1]
    # Multiply by 100 to convert the 0–1 scale to percentage points (pp),
    # which is how practitioners talk about recall/precision changes.
    d_recall = (new["recall_at_0_5"] - base["recall_at_0_5"]) * 100
    d_prec   = (new["precision_at_0_5"] - base["precision_at_0_5"]) * 100
    # Alert count is a raw integer, so the delta is just subtraction.
    d_alerts = int(new["alert_count_at_0_5"] - base["alert_count_at_0_5"])
    print(f"Recall    : {'↑' if d_recall >= 0 else '↓'} {abs(d_recall):.1f} pp  ({base['recall_at_0_5']:.3f} → {new['recall_at_0_5']:.3f})")
    print(f"Precision : {'↑' if d_prec >= 0 else '↓'} {abs(d_prec):.1f} pp  ({base['precision_at_0_5']:.3f} → {new['precision_at_0_5']:.3f})")
    print(f"Alerts    : {'↑' if d_alerts >= 0 else '↓'} {abs(d_alerts):,}  ({int(base['alert_count_at_0_5']):,} → {int(new['alert_count_at_0_5']):,})")


print_improvement(comparison_df)

         model    pr_auc  recall_at_0_5  precision_at_0_5  alert_count_at_0_5
0  rf_balanced  0.883456       0.726807          0.957617                1628
1     rf_smote  0.865872       0.789744          0.860335                1969
Recall    : ↑ 6.3 pp  (0.727 → 0.790)
Precision : ↓ 9.7 pp  (0.958 → 0.860)
Alerts    : ↑ 341  (1,628 → 1,969)


## Step 9: Threshold Calibration

### What is a decision threshold?

The model outputs a **probability** between 0 and 1 for each transaction: "there is a 73% chance this is fraud." The **threshold** is the line you draw — above it, the transaction gets flagged as fraud; below it, it doesn't.

The default threshold of **0.5** was designed for datasets where classes are roughly equal. On this dataset, only 0.58% of transactions are fraudulent. A fraud score of 0.3 is already ~52× the base rate — yet the default threshold would let it pass unchallenged.

### The precision–recall see-saw

Lowering the threshold catches more fraud (↑ recall) but also generates more false alerts (↓ precision). Raising it does the opposite. You cannot improve both at once — every operating point is a tradeoff:

```
Low threshold  →  catch everything  →  many false alarms  (high recall, low precision)
High threshold →  only sure cases   →  miss some fraud    (low recall, high precision)
```

### Business framing

A bank might set a **Service Level Agreement (SLA)**: "catch at least 85% of frauds, then minimise false alerts." This translates directly to a constraint on the PR curve: find the threshold that achieves `recall ≥ 0.85`, then pick the one with the highest precision at that recall level.

**F1-optimal threshold**: maximises the harmonic mean of precision and recall — a good default when no specific SLA exists.

### What to expect

The calibrated threshold will be lower than 0.5. Recall will increase significantly; precision may drop somewhat. The goal is to find the operating point that matches business priorities.

In [12]:
from sklearn.metrics import precision_recall_curve, precision_score, recall_score

# smote_artifacts stores everything evaluate_binary_model() computed for rf_smote,
# including the raw probability scores for every test transaction.
# We use these scores (not the hard 0/1 predictions) because threshold calibration
# works by moving the cutoff on the continuous score — not re-running the model.
best_scores = smote_artifacts["y_score_test"]

# precision_recall_curve() sweeps every possible threshold from 0 to 1 and
# records what precision and recall would be at each one. This gives us the
# full picture of all possible operating points before we pick one.
# Note: len(thresholds) == len(precisions) - 1 because sklearn adds a sentinel
# value at the end of precisions/recalls — that's why we use [:-1] below.
precisions, recalls, thresholds = precision_recall_curve(y_test_np, best_scores)

# F1 is the harmonic mean of precision and recall — it penalises extreme
# imbalances between the two. argmax() finds the index where F1 is highest,
# and we look up the corresponding threshold value.
# The +1e-9 prevents division by zero if both precision and recall are 0.
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-9)
best_thresh_idx = f1_scores.argmax()
best_threshold = thresholds[best_thresh_idx]

# np.where() returns the indices where the condition is True — i.e., every
# threshold that achieves recall >= 85%. We take the last one ([-1]) because
# thresholds are sorted ascending: the last qualifying index gives the highest
# threshold that still meets the SLA, which also maximises precision.
recall_85_idxs = np.where(recalls[:-1] >= 0.85)[0]
thresh_85_recall = thresholds[recall_85_idxs[-1]] if len(recall_85_idxs) > 0 else best_threshold

for thresh_name, thresh_val in [("max_f1", best_threshold), ("recall_85", thresh_85_recall)]:
    # Convert continuous scores to hard 0/1 labels using the chosen threshold.
    # astype(int) turns the boolean array (True/False) into integers (1/0).
    y_pred = (best_scores >= thresh_val).astype(int)
    prec = precision_score(y_test_np, y_pred, zero_division=0)
    rec  = recall_score(y_test_np, y_pred)
    alerts = int(y_pred.sum())  # total transactions flagged as fraud
    print(f"[{thresh_name}]  threshold={thresh_val:.4f}")
    print(f"  precision={prec:.3f}  recall={rec:.3f}  alerts={alerts:,}")
    print()

# Banks typically prioritise recall over precision: the cost of a missed fraud
# (full transaction loss + reputational damage) far exceeds the cost of
# investigating a false alarm (a quick customer phone call).
print("Recommendation: prefer recall_85 — missing fraud is far costlier than a false alert.")

[max_f1]  threshold=0.5267
  precision=0.877  recall=0.781  alerts=1,910

[recall_85]  threshold=0.3033
  precision=0.703  recall=0.850  alerts=2,594

Recommendation: prefer recall_85 — missing fraud is far costlier than a false alert.


## Step 10: Random Forest Hyperparameter Tuning

### Parameters vs hyperparameters

A model has two kinds of settings:
- **Parameters** — what the model *learns* from data: the specific split conditions in each tree, the thresholds at each node.
- **Hyperparameters** — what *you set* before training: how many trees to build, how deep they can grow, how many features to consider at each split.

Hyperparameters cannot be learned from the training data directly — they control the learning process itself.

### Manual tuning approach

Instead of running an expensive automated search (e.g. RandomizedSearchCV with 30+ fits on 1M+ rows), we pick sensible hyperparameters directly:

| Hyperparameter | Value | Rationale |
|---|---|---|
| `n_estimators=300` | More trees = more stable predictions, diminishing returns beyond ~300 |
| `max_depth=30` | Deep enough to capture complex fraud patterns without being unlimited |
| `min_samples_leaf=2` | Slight regularisation to avoid memorising single outliers |
| `max_features="sqrt"` | Classic default — decorrelates trees by limiting features per split |
| `class_weight="balanced_subsample"` | Per-tree class reweighting complements SMOTE |

### What to expect

A small improvement over SMOTE RF — roughly 1-3% PR-AUC. The bigger gains came from SMOTE and threshold calibration; tuning is the final polish.

In [ ]:
rf_tuned = RandomForestClassifier(
    n_estimators=300,
    max_depth=30,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_tuned.fit(X_train_sm, y_train_sm)

tuned_metrics, tuned_artifacts = evaluate_binary_model(
    "rf_tuned", rf_tuned,
    X_train_sm, y_train_sm, X_test_np, y_test_np,
    threshold=best_threshold
)

final_df = pd.DataFrame([baseline_results_df.iloc[0].to_dict(), smote_metrics, tuned_metrics])
print(final_df[["model", "pr_auc", "recall_at_0_5", "precision_at_0_5", "alert_count_at_0_5"]])

## Step 11: Validation & Reliability Check

> **Goal:** Answer one question — *"Should I trust this model?"*
> PR-AUC of 0.90 sounds great, but a single number can hide serious problems. These three checks make hidden problems visible.

---

### Check 1 — Confusion Matrix (raw counts)
A metric like PR-AUC compresses everything into one number. The confusion matrix shows **what's actually happening**: how many frauds are caught (TP), how many are missed (FN), and how many legitimate transactions are incorrectly flagged (FP).

### Check 2 — Overfitting (train PR-AUC vs test PR-AUC)
If the model scores much higher on training data than on test data, it **memorised** the training examples rather than learning general patterns. A gap > ~0.05 is a warning sign.

### Check 3 — Feature Importances
If the model is learning real fraud signals, meaningful features (transaction amount, hour of day, category risk) should rank high.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, average_precision_score
import matplotlib.pyplot as plt

tuned_model = tuned_artifacts["model"]
# Apply the calibrated threshold from Step 9 (not the default 0.5)
y_pred_tuned = (tuned_artifacts["y_score_test"] >= best_threshold).astype(int)

# -- Check 1: Confusion matrix ------------------------------------------------
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test_np, y_pred_tuned, ax=ax,
    display_labels=["Legitimate", "Fraud"], colorbar=False
)
ax.set_title(f"Confusion Matrix — rf_tuned (threshold={best_threshold:.3f})")
plt.tight_layout()
plt.show()

# -- Check 2: Overfitting -----------------------------------------------------
train_score_tuned = average_precision_score(
    y_train_sm, tuned_model.predict_proba(X_train_sm)[:, 1]
)
test_score_tuned = average_precision_score(
    y_test_np, tuned_artifacts["y_score_test"]
)
print(f"Overfitting check — PR-AUC:  train={train_score_tuned:.4f}  test={test_score_tuned:.4f}")
gap = train_score_tuned - test_score_tuned
print(f"  Gap (train - test) = {gap:.4f} {'<- possible overfit' if gap > 0.05 else '<- acceptable'}")

# -- Check 3: Feature importances ---------------------------------------------
importances = pd.Series(tuned_model.feature_importances_, index=FEATURE_COLS)
importances = importances.sort_values(ascending=False)
print(f"\nTop feature importances (rf_tuned):")
print(importances.head(10).to_string())

## Step 12: GPU Configuration

### Why GPU?
Tree models (Random Forest) run on CPU. But neural networks perform millions of matrix multiplications per batch — GPUs have thousands of cores optimised for exactly this. A Keras DNN that takes minutes on CPU can train in seconds on GPU.

### Mixed precision (`mixed_float16`)
Modern NVIDIA GPUs (RTX series) have **Tensor Cores** that process float16 math ~2x faster than float32. Mixed precision keeps model weights in float32 for numerical stability but runs forward/backward passes in float16 for speed. The loss scaling is handled automatically by Keras.

In [ ]:
# ---------------------------------------------------------------------------
# Step 12: GPU configuration
# ---------------------------------------------------------------------------
# Detect available GPUs and configure TensorFlow for efficient GPU usage.

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print(f"GPU(s) detected: {len(gpus)}")
    for gpu in gpus:
        print(f"  - {gpu.name}")
        # Memory growth: allocate VRAM incrementally instead of grabbing all at once.
        # Without this, TF reserves the entire GPU memory on startup, which can
        # crash other applications or prevent running multiple notebooks.
        tf.config.experimental.set_memory_growth(gpu, True)

    # Mixed precision: use float16 for computation, float32 for variables.
    # This roughly doubles throughput on RTX/Tensor Core GPUs.
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision enabled (mixed_float16)")
else:
    print("WARNING: No GPU detected. Training will run on CPU (much slower).")
    print("Check: is tensorflow[and-cuda] installed? Is your NVIDIA driver up to date?")

> **Colab Note:** cuML (RAPIDS) requires special setup on Colab. The cell below will gracefully skip if cuML is not installed. See the [RAPIDS Colab guide](https://docs.rapids.ai/deployment/stable/platforms/colab/) for installation instructions.

## Step 13: GPU-Accelerated Random Forest (cuML)

### What is cuML?
cuML is part of NVIDIA's RAPIDS suite. It provides GPU-accelerated versions of scikit-learn algorithms — same API, same results, but running on CUDA cores instead of CPU threads.

### Why use it here?
The sklearn Random Forest in Step 10 trained on ~2.6M rows (SMOTE-expanded). On CPU with `n_jobs=-1`, this takes several minutes. The cuML version uses the GPU to parallelise tree construction, typically achieving 10-50x speedup.

### Expected results
PR-AUC should be very similar to the sklearn RF (~0.88-0.90). The algorithm is the same — only the hardware changes. Any small differences come from floating-point ordering on GPU vs CPU.

In [ ]:
# ---------------------------------------------------------------------------
# Step 13: GPU-accelerated Random Forest (cuML)
# ---------------------------------------------------------------------------
import time

try:
    from cuml.ensemble import RandomForestClassifier as cuRF

    print("cuML imported successfully — training GPU Random Forest...")
    t0 = time.perf_counter()

    # cuML RF accepts NumPy arrays directly (converts to GPU internally).
    # Note: cuML RF does not support class_weight, so we train on SMOTE data
    # which already handles the imbalance via synthetic oversampling.
    rf_gpu = cuRF(
        n_estimators=300,
        max_depth=30,
        max_features="sqrt",
        random_state=RANDOM_STATE,
    )
    rf_gpu.fit(X_train_sm.astype(np.float32), y_train_sm.astype(np.int32))

    gpu_time = time.perf_counter() - t0
    print(f"cuML RF training time: {gpu_time:.1f}s")

    # Evaluate using same metrics as sklearn models
    y_score_gpu = rf_gpu.predict_proba(X_test_np.astype(np.float32))[:, 1]
    y_pred_gpu = (np.array(y_score_gpu) >= best_threshold).astype(int)

    gpu_roc = roc_auc_score(y_test_np, np.array(y_score_gpu))
    gpu_pr = average_precision_score(y_test_np, np.array(y_score_gpu))
    gpu_prec = precision_score(y_test_np, y_pred_gpu, zero_division=0)
    gpu_rec = recall_score(y_test_np, y_pred_gpu, zero_division=0)
    gpu_f1 = f1_score(y_test_np, y_pred_gpu, zero_division=0)

    cuml_metrics = {
        "model": "rf_gpu_cuml",
        "roc_auc": gpu_roc,
        "pr_auc": gpu_pr,
        "precision_at_0_5": gpu_prec,
        "recall_at_0_5": gpu_rec,
        "f1_at_0_5": gpu_f1,
        "alert_count_at_0_5": int(y_pred_gpu.sum()),
        "alert_rate_at_0_5": float(y_pred_gpu.sum() / len(y_pred_gpu)),
    }
    print(f"\ncuML RF — ROC-AUC: {gpu_roc:.4f}  PR-AUC: {gpu_pr:.4f}")
    print(f"  Precision: {gpu_prec:.4f}  Recall: {gpu_rec:.4f}  F1: {gpu_f1:.4f}")
    CUML_AVAILABLE = True

except ImportError:
    print("cuML not available — skipping GPU Random Forest.")
    print("Install via: conda install -c rapidsai cuml")
    cuml_metrics = None
    CUML_AVAILABLE = False

## Step 14: Keras Deep Neural Network (GPU)

### Why a neural network for tabular fraud data?
Tree ensembles (Random Forest) are strong on tabular data, but neural networks can learn different feature interactions — especially useful when combined in an ensemble. The DNN also serves as a benchmark: if it can't beat the tuned RF, the data likely doesn't have complex non-linear patterns that trees miss.

### Architecture decisions
The `build_fraud_dnn()` function below defines the network architecture. Key design choices:
- **Layer sizes**: start wide (128), narrow toward the output — a "funnel" that compresses information
- **BatchNormalization**: stabilises training under extreme class imbalance and mixed precision (float16)
- **Dropout**: regularisation that randomly disables neurons during training to prevent overfitting
- **Class weights** instead of SMOTE: for neural nets, reweighting the loss is preferred over synthetic samples (avoids the network memorising SMOTE artefacts)
- **Sigmoid output**: outputs a probability between 0 and 1, matching our threshold calibration approach

In [ ]:
# ---------------------------------------------------------------------------
# Step 14: Keras DNN — model architecture
# ---------------------------------------------------------------------------

def build_fraud_dnn(n_features):
    """Build a Keras DNN for binary fraud classification.

    TODO(human): Implement the model architecture below.

    Suggested starting point (feel free to change):
      Input(n_features) -> Dense(128, relu) -> BatchNorm -> Dropout(0.3)
                        -> Dense(64, relu)  -> BatchNorm -> Dropout(0.3)
                        -> Dense(32, relu)  -> Dropout(0.2)
                        -> Dense(1, sigmoid)

    Considerations:
      - 11 input features: don't go too wide (512+) or you'll overfit
      - BatchNorm helps with mixed_float16 and extreme class imbalance
      - Dropout should decrease toward the output layer
      - Final layer MUST be Dense(1, activation="sigmoid", dtype="float32")
        (float32 is required for numerical stability with mixed precision)
    """
    inputs = layers.Input(shape=(n_features,))

    # ---- YOUR ARCHITECTURE HERE ----
    # Build hidden layers using: layers.Dense, layers.BatchNormalization, layers.Dropout
    # Example for one block:
    #   x = layers.Dense(128, activation="relu")(inputs)
    #   x = layers.BatchNormalization()(x)
    #   x = layers.Dropout(0.3)(x)

    raise NotImplementedError("TODO(human): implement the hidden layers above")

    # Output: single sigmoid neuron (fraud probability)
    # dtype="float32" ensures stable gradient computation under mixed precision
    outputs = layers.Dense(1, activation="sigmoid", dtype="float32")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model


# Quick sanity check: build and print summary
n_features = X_train_np.shape[1]
print(f"Building DNN for {n_features} input features...")
dnn_model = build_fraud_dnn(n_features)
dnn_model.summary()

In [ ]:
# ---------------------------------------------------------------------------
# Step 14b: Keras DNN — compile and train
# ---------------------------------------------------------------------------

# Class weights: tell the loss function that missing a fraud is ~172x worse
# than a false alarm. This replaces SMOTE for the neural network.
n_fraud = y_train_np.sum()
n_legit = len(y_train_np) - n_fraud
fraud_weight = n_legit / n_fraud
dnn_class_weights = {0: 1.0, 1: float(fraud_weight)}
print(f"Class weights: legit=1.0, fraud={fraud_weight:.1f}")

dnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.AUC(name="roc_auc", curve="ROC"),
        keras.metrics.AUC(name="pr_auc", curve="PR"),
    ],
)

# Callbacks:
# - EarlyStopping: stop training if val_pr_auc hasn't improved for 5 epochs
#   (prevents overfitting and saves time)
# - ReduceLROnPlateau: halve the learning rate if val_pr_auc stalls for 3 epochs
#   (fine-tunes convergence in later epochs)
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_pr_auc", patience=5, mode="max", restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_pr_auc", patience=3, factor=0.5, mode="max", verbose=1
    ),
]

# Cast to float32: StandardScaler outputs float64, but GPU mixed precision
# expects float32 inputs (it then internally uses float16 for speed).
X_train_f32 = X_train_np.astype(np.float32)
X_test_f32 = X_test_np.astype(np.float32)

print(f"\nTraining DNN on {len(X_train_f32):,} samples...")
print(f"Validation on {len(X_test_f32):,} samples")

history = dnn_model.fit(
    X_train_f32, y_train_np,
    validation_data=(X_test_f32, y_test_np),
    epochs=50,
    batch_size=2048,       # large batch saturates GPU; 11 features = tiny memory per sample
    class_weight=dnn_class_weights,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# ---------------------------------------------------------------------------
# Step 14c: Keras DNN — evaluation and model comparison
# ---------------------------------------------------------------------------

# Predict fraud probabilities on test set
y_score_dnn = dnn_model.predict(X_test_f32, batch_size=4096).ravel()
y_pred_dnn = (y_score_dnn >= best_threshold).astype(int)

dnn_roc = roc_auc_score(y_test_np, y_score_dnn)
dnn_pr = average_precision_score(y_test_np, y_score_dnn)
dnn_prec = precision_score(y_test_np, y_pred_dnn, zero_division=0)
dnn_rec = recall_score(y_test_np, y_pred_dnn, zero_division=0)
dnn_f1 = f1_score(y_test_np, y_pred_dnn, zero_division=0)

dnn_metrics = {
    "model": "keras_dnn",
    "roc_auc": dnn_roc,
    "pr_auc": dnn_pr,
    "precision_at_0_5": dnn_prec,
    "recall_at_0_5": dnn_rec,
    "f1_at_0_5": dnn_f1,
    "alert_count_at_0_5": int(y_pred_dnn.sum()),
    "alert_rate_at_0_5": float(y_pred_dnn.sum() / len(y_pred_dnn)),
}

print(f"Keras DNN — ROC-AUC: {dnn_roc:.4f}  PR-AUC: {dnn_pr:.4f}")
print(f"  Precision: {dnn_prec:.4f}  Recall: {dnn_rec:.4f}  F1: {dnn_f1:.4f}")

# -- Unified comparison table --------------------------------------------------
all_metrics = [
    baseline_results_df.iloc[0].to_dict(),  # best sklearn baseline
    smote_metrics,
    tuned_metrics,
    dnn_metrics,
]
if cuml_metrics is not None:
    all_metrics.insert(3, cuml_metrics)  # insert before DNN

comparison_df = pd.DataFrame(all_metrics).sort_values("pr_auc", ascending=False).reset_index(drop=True)
print("\nAll models comparison (sorted by PR-AUC):")
print(comparison_df[["model", "roc_auc", "pr_auc", "precision_at_0_5", "recall_at_0_5", "f1_at_0_5", "alert_count_at_0_5"]].to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------
# Step 14d: Training history visualization
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss (binary crossentropy)")
axes[0].set_xlabel("Epoch")
axes[0].legend()

# ROC-AUC
axes[1].plot(history.history["roc_auc"], label="train")
axes[1].plot(history.history["val_roc_auc"], label="val")
axes[1].set_title("ROC-AUC")
axes[1].set_xlabel("Epoch")
axes[1].legend()

# PR-AUC
axes[2].plot(history.history["pr_auc"], label="train")
axes[2].plot(history.history["val_pr_auc"], label="val")
axes[2].set_title("PR-AUC (primary metric)")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.tight_layout()
plt.show()

# Print best epoch info
best_epoch = np.argmax(history.history["val_pr_auc"])
print(f"Best epoch: {best_epoch + 1}")
print(f"  val_pr_auc = {history.history['val_pr_auc'][best_epoch]:.4f}")
print(f"  val_roc_auc = {history.history['val_roc_auc'][best_epoch]:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 14e: Save Keras model
# ---------------------------------------------------------------------------
# .keras is the recommended format (TF 2.16+). It stores architecture,
# weights, and optimizer state in a single file.
dnn_model.save("fraud_dnn_model.keras")
print("Keras DNN saved to fraud_dnn_model.keras")

# Save the sklearn tuned RF alongside for comparison
import joblib
joblib.dump(tuned_artifacts["model"], "fraud_rf_tuned.joblib")
print("Tuned RF saved to fraud_rf_tuned.joblib")